# Risk Prediction Notebook: RandomForest


- Same feature list
- Same boolean conversion (`True -> 1.0`, `False -> 0.0`, unknown -> `0.5`)
- Same model settings: `RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)`


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

FEATURES = [
    'antivirus_detected', 'antivirus_enabled', 'antivirus_up_to_date',
    'disk_encrypted', 'patch_is_current', 'usb_enabled',
    'ram_total_mb', 'disk_total_gb', 'disk_free_gb',
    'local_port_count', 'lan_device_count', 'wifi_open_network_count',
    'patch_days_since_update',
]

LEVELS = ['low', 'medium', 'high', 'critical']
print('Feature count:', len(FEATURES))


In [ ]:
def bool_int(val):
    if val is True:
        return 1.0
    if val is False:
        return 0.0
    return 0.5

def snapshot_to_row(snapshot: dict):
    return [
        bool_int(snapshot.get('antivirus_detected')),
        bool_int(snapshot.get('antivirus_enabled')),
        bool_int(snapshot.get('antivirus_up_to_date')),
        bool_int(snapshot.get('disk_encrypted')),
        bool_int(snapshot.get('patch_is_current')),
        bool_int(snapshot.get('usb_enabled')),
        float(snapshot.get('ram_total_mb') or 0),
        float(snapshot.get('disk_total_gb') or 0),
        float(snapshot.get('disk_free_gb') or 0),
        float(snapshot.get('local_port_count') or 0),
        float(snapshot.get('lan_device_count') or 0),
        float(snapshot.get('wifi_open_network_count') or 0),
        float(snapshot.get('patch_days_since_update') or 0),
    ]


In [ ]:
# Optional real-data loader: set CSV_PATH to your exported snapshot CSV.
CSV_PATH = ''  # e.g., '../Deliverables/snapshots_export.csv'

if CSV_PATH:
    df = pd.read_csv(CSV_PATH)
    print('Loaded real dataset:', df.shape)
else:
    # Synthetic fallback dataset for runnable demo
    rng = np.random.default_rng(42)
    rows = []
    for i in range(80):
        row = {
            'antivirus_detected': bool(rng.integers(0, 2)),
            'antivirus_enabled': bool(rng.integers(0, 2)),
            'antivirus_up_to_date': bool(rng.integers(0, 2)),
            'disk_encrypted': bool(rng.integers(0, 2)),
            'patch_is_current': bool(rng.integers(0, 2)),
            'usb_enabled': bool(rng.integers(0, 2)),
            'ram_total_mb': int(rng.integers(4096, 65536)),
            'disk_total_gb': int(rng.integers(128, 2048)),
            'disk_free_gb': int(rng.integers(8, 1024)),
            'local_port_count': int(rng.integers(5, 180)),
            'lan_device_count': int(rng.integers(1, 40)),
            'wifi_open_network_count': int(rng.integers(0, 8)),
            'patch_days_since_update': int(rng.integers(0, 90)),
        }

        risk_score = (
            (0 if row['antivirus_enabled'] else 1)
            + (0 if row['disk_encrypted'] else 1)
            + min(row['patch_days_since_update'] / 30.0, 3.0)
            + (row['wifi_open_network_count'] / 3.0)
        )
        if risk_score < 1.5:
            row['risk_level'] = 'low'
        elif risk_score < 2.5:
            row['risk_level'] = 'medium'
        elif risk_score < 3.5:
            row['risk_level'] = 'high'
        else:
            row['risk_level'] = 'critical'

        rows.append(row)

    df = pd.DataFrame(rows)
    print('Generated synthetic dataset:', df.shape)

df.head()


In [ ]:
X_train = np.array([snapshot_to_row(rec) for rec in df.to_dict(orient='records')])
y_train = df['risk_level'].astype(str).values

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

print('Training complete')
print('Classes:', list(clf.classes_))


In [ ]:
latest_snapshot = {
    'antivirus_detected': True,
    'antivirus_enabled': False,
    'antivirus_up_to_date': False,
    'disk_encrypted': False,
    'patch_is_current': False,
    'usb_enabled': True,
    'ram_total_mb': 8192,
    'disk_total_gb': 512,
    'disk_free_gb': 60,
    'local_port_count': 95,
    'lan_device_count': 18,
    'wifi_open_network_count': 3,
    'patch_days_since_update': 45,
}

X_pred = np.array([snapshot_to_row(latest_snapshot)])
predicted_level = clf.predict(X_pred)[0]
proba = clf.predict_proba(X_pred)[0]
classes = list(clf.classes_)
confidence_pct = round(float(proba[classes.index(predicted_level)]) * 100, 1)

print('Predicted risk level:', predicted_level)
print('Confidence %:', confidence_pct)
print('Class probabilities:', {cls: round(float(p) * 100, 1) for cls, p in zip(classes, proba)})


In [ ]:
importances = [
    {'feature': name, 'importance': round(float(imp), 4)}
    for name, imp in zip(FEATURES, clf.feature_importances_)
]
importances = sorted(importances, key=lambda x: -x['importance'])
pd.DataFrame(importances[:8])
